In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, TimeDistributed

def generate_series(n=1000):
    t = np.arange(n)
    return 10 + 0.01 * t + np.sin(2 * np.pi * t / 50) + 0.1 * np.random.randn(n)

series = generate_series(1000)
scaler = StandardScaler()
series_scaled = scaler.fit_transform(series.reshape(-1, 1)).flatten()

INPUT_WINDOW, OUTPUT_WINDOW = 30, 10

def make_windows(data, input_w, output_w):
    X, Y = [], []
    for i in range(len(data) - input_w - output_w):
        X.append(data[i:i + input_w])
        Y.append(data[i + input_w:i + input_w + output_w])
    return np.array(X), np.array(Y)

X, Y = make_windows(series_scaled, INPUT_WINDOW, OUTPUT_WINDOW)
X, Y = X[..., np.newaxis], Y[..., np.newaxis]

split = int(0.8 * len(X))
X_train, X_test, Y_train, Y_test = X[:split], X[split:], Y[:split], Y[split:]

encoder_inputs = Input(shape=(INPUT_WINDOW, 1))
_, state_h, state_c = LSTM(64, return_state=True)(encoder_inputs)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(OUTPUT_WINDOW, 1))
decoder_outputs, _, _ = LSTM(64, return_sequences=True, return_state=True)(decoder_inputs, initial_state=encoder_states)
decoder_outputs = TimeDistributed(Dense(1))(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

decoder_input_train, decoder_input_test = np.zeros_like(Y_train), np.zeros_like(Y_test)

model.fit([X_train, decoder_input_train], Y_train, validation_data=([X_test, decoder_input_test], Y_test), epochs=20, batch_size=32, verbose=1)

preds = model.predict([X_test, decoder_input_test])
preds_unscaled = scaler.inverse_transform(preds.reshape(-1, 1)).reshape(preds.shape)
Y_test_unscaled = scaler.inverse_transform(Y_test.reshape(-1, 1)).reshape(Y_test.shape)

mae = mean_absolute_error(Y_test_unscaled.flatten(), preds_unscaled.flatten())
rmse = np.sqrt(mean_squared_error(Y_test_unscaled.flatten(), preds_unscaled.flatten()))
print(f"Test MAE: {mae:.4f}, Test RMSE: {rmse:.4f}")

Epoch 1/20


c:\Users\desai\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor', 'keras_tensor_4']. Received: the structure of inputs=('*', '*')
  warnings.warn(


24/24 ━━━━━━━━━━━━━━━━━━━━ 10s 69ms/step - loss: 0.3975 - mae: 0.5208 - val_loss: 0.4574 - val_mae: 0.6196
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0840 - mae: 0.2462 - val_loss: 0.2007 - val_mae: 0.3834
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0651 - mae: 0.2195 - val_loss: 0.0962 - val_mae: 0.2497
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0571 - mae: 0.2027 - val_loss: 0.0904 - val_mae: 0.2448
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0435 - mae: 0.1771 - val_loss: 0.0852 - val_mae: 0.2411
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0214 - mae: 0.1159 - val_loss: 0.0383 - val_mae: 0.1672
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0058 - mae: 0.0587 - val_loss: 0.0462 - val_mae: 0.1803
Epoch 8/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0038 - mae: 0.0487 - val_loss: 0.0502 - val_mae: 0.1972
Epoch 9/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0036 - mae: 0